# Example loading a batch of data using the PyTorch dataloader

This notebook is intended to run in [Google Colab](https://colab.research.google.com/github/FrontierDevelopmentLab/2025-HL-Ionosphere-dataset/blob/main/dataset_example_colab.ipynb), which has most of the dependencies pre-installed. 

The dataset is organized as follows (using wildcard/glob notation, as well as parentheses around year/month/day variables):

```
ionosphere-data-public
├── celestrak
│   ├── celestrak_SW-All.csv
│   ├── kp_ap_processed_timeseries.csv
│   └── kp_ap_processed_timeseries_deltamin_15_rewind_180.csv
├── google_android
│   ├── CAS0OPSRAP_20233090000_01D_01D_DCB.BIA.gz
│   ├── dcb_clusters.csv.gz
│   ├── LICENSE
│   ├── sat_xyz.csv.gz
│   ├── TLSE00FRA_R_20233090000_01D_30S_MO.crx.gz
│   ├── video_2023_09_11_to_2024_05_24.mp4
│   ├── vtec_maps
│   └── worldpop_population_per_km2_s2level10.csv.gz
├── jpld
│   ├── raw
│   │   └── (year)
│   │       └── jpld*.10i.nc.gz
│   └── webdataset
│       ├── dates_index_*
│       ├── jpld-*.tar
│       └── tar_files_index_3a8334f288afd9deb25c6ec57c6b0cbb
├── madrigal
│   ├── gps_tec_20120216.txt
│   ├── processed
│   │   ├── gps_data  
│   │   │   └── (2-digit year, e.g., 09)
│   │   │       └── gps_data_(4-digit-year).txt
│   │   └── gps_data_tarr
│   │       ├── (year)
│   │       │   └── (2-digit-month)
│   │       │       └── (2-digit-day)
│   │       │           └── *.tec_madrigal.npy
│   │       ├── csv_subsets
│   │       │   └── subset_tec_?0mln.csv
│   │       └── tars
│   │           └── dataset-*.tar
│   └── raw
│       ├── gps_data
│       │   └── gps*g.001.hdf5
│       ├── ne_dir
│       │   └── pfa*.001_lp_fit_*min.001.h5.gz
│       └── tec_dir
│           └── pfa*.001_lp_fit_*min.001.h5.gz
├── omniweb
│   ├── omniweb_indices_15min.csv
│   ├── omniweb_magnetic_field_15min.csv
│   └── omniweb_solar_wind_15min.csv
├── quasidipole
│   └── qd_lon_(year).npy
├── README.md
├── sdocore
│   └── sdo_core_dataset_21504.h5
└── set
    ├── karman-2025_data_sw_data_set_sw.csv
    └── karman-2025_data_sw_data_set_sw_deltamin_15_rewind_1440.csv
```

In [ ]:
!pip install cartopy awscli skyfield
!pip install pandas<3

In [ ]:
# Download the data processing scripts
!git clone --filter=blob:none --sparse https://github.com/FrontierDevelopmentLab/2025-HL-Ionosphere-dataset.git 
!cd 2025-HL-Ionosphere-dataset && git sparse-checkout set scripts/
!sleep 2  # wait 2 seconds 
!mv 2025-HL-Ionosphere-dataset/scripts .
!rmdir 2025-HL-Ionosphere-dataset

### Imports

In [ ]:
import os
import torch
import pickle
import hashlib
import tarfile
import datetime
import warnings
import numpy as np
import pandas as pd
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import skyfield.api
from tqdm import tqdm
from glob import glob
from io import BytesIO
from functools import lru_cache
from torch.utils.data import Dataset

In [ ]:
from scripts.events.events import EventCatalog

from scripts.datasets.dataset_jpld import JPLD
from scripts.datasets.dataset_sequences import Sequences
from scripts.datasets.dataset_union import Union
from scripts.datasets.dataset_sunmoongeometry import SunMoonGeometry
from scripts.datasets.dataset_quasidipole import QuasiDipole
from scripts.datasets.dataset_celestrak import CelesTrak
from scripts.datasets.dataset_omniweb import OMNIWeb, omniweb_all_columns
from scripts.datasets.dataset_set import SET, set_all_columns
from scripts.datasets.dataloader_cached import CachedDataLoader

## Stream the dataset from AWS

In [ ]:
# Load Omniweb and JPLD data (only two JPLD files corresponding to end of the dataset)
!mkdir -p data/jpld
!mkdir -p data/omniweb/

!aws s3 sync --no-sign-request s3://nasa-radiant-data/helioai-datasets/ionosphere-data-public/omniweb/ data/omniweb/

!aws s3 cp --no-sign-request s3://nasa-radiant-data/helioai-datasets/ionosphere-data-public/jpld/webdataset/tar_files_index_3a8334f288afd9deb25c6ec57c6b0cbb data/jpld/tar_files_index_3a8334f288afd9deb25c6ec57c6b0cbb
!aws s3 cp --no-sign-request s3://nasa-radiant-data/helioai-datasets/ionosphere-data-public/jpld/webdataset/jpld-168.tar data/jpld/jpld-168.tar
!aws s3 cp --no-sign-request s3://nasa-radiant-data/helioai-datasets/ionosphere-data-public/jpld/webdataset/jpld-169.tar data/jpld/jpld-169.tar
!aws s3 cp --no-sign-request s3://nasa-radiant-data/helioai-datasets/ionosphere-data-public/jpld/webdataset/jpld-170.tar data/jpld/jpld-170.tar
!aws s3 cp --no-sign-request s3://nasa-radiant-data/helioai-datasets/ionosphere-data-public/jpld/webdataset/jpld-171.tar data/jpld/jpld-171.tar

## Load a single batch of data
Example of just loading a single batch of JPLD data and plotting it

In [ ]:
# Create a dataset
dataset_jpld_dir = 'data/jpld'

date_start = datetime.datetime(2024, 5, 9)
date_end = datetime.datetime(2024, 5, 10)

dataset_jpld = JPLD(dataset_jpld_dir, date_start=date_start, date_end=date_end)

In [ ]:
# Load the JPLD map corresponding to date datetime.datetime(2024, 5, 9, 18, 30)
date = datetime.datetime(2024, 5, 9, 18, 30)
tec_map, tec_date = dataset_jpld[date]
print(tec_map.shape, tec_date)

In [ ]:
# Plot the TEC map
# Create a GeoAxes instance with a specific projection
fig = plt.figure(figsize=(8, 4))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

im = ax.imshow(
    tec_map[0],
    extent=[-180, 180, -90, 90],
    origin='upper',
    cmap='jet',
    transform=ccrs.PlateCarree()
)

ax.coastlines()

plt.show()

## Sequence of multiple datasets setup
Example of setting up a sequence dataset for with the JPLD, OMNIWeb, and the SunMoon datasets

In [ ]:
# Initialize variables
date_start = datetime.datetime(2024, 5, 9)
date_end = datetime.datetime(2024, 5, 10)
image_size = (180, 360)
context_window = 2 # Number of time steps of context used in model
prediction_window = 1 # Number of time steps to predict
training_sequence_length = context_window + prediction_window
delta_minutes = 15 # 15-minute cadence
date_dilation = 16 # Use 16x dilation for 15-min data to get 4-hour context

date_exclusions = []
datasets_omniweb_valid = []
datasets_jpld_valid = []
datasets_sunmoon_valid = []

dataset_omniweb_dir = 'data/omniweb/' #'/path/to/omniweb/data/'
dataset_jpld_dir = 'data/jpld' #'/path/to/jpld/data/'

# Create sequence dataset excluding validation events
dataset_jpld = JPLD(dataset_jpld_dir, date_start=date_start, date_end=date_end, date_exclusions=date_exclusions)
dataset_omniweb = OMNIWeb(dataset_omniweb_dir, date_start=date_start, date_end=date_end, date_exclusions=date_exclusions, return_as_image_size=image_size)
dataset_sunmoon = SunMoonGeometry(date_start=date_start, date_end=date_end, date_exclusions=date_exclusions)

# Create Sequence datasets
dataset = Sequences(datasets=[dataset_jpld, dataset_omniweb, dataset_sunmoon], sequence_length=training_sequence_length, dilation=date_dilation, delta_minutes=delta_minutes)


## Train and Validation setup

Example of setting up a dataset for training and validation using the JPLD, OMNIWeb, and the SunMoon datasets, excluding certain events for validation.

In [ ]:
# Initialize variables
date_start = datetime.datetime(2024, 5, 9)
date_end = datetime.datetime(2024, 5, 10)
image_size = (180, 360)
context_window = 2 # Number of time steps of context used in model
prediction_window = 1 # Number of time steps to predict
training_sequence_length = context_window + prediction_window
delta_minutes = 15 # 15-minute cadence
date_dilation = 16 # Use 16x dilation for 15-min data to get 4-hour context

dataset_omniweb_dir = 'data/omniweb/' #'/path/to/omniweb/data/'
dataset_jpld_dir = 'data/jpld' #'/path/to/jpld/data/'

# Make a EventCatalog[event_id], a dict with keys:
# 'date_start': date_start,
# 'date_end': date_end,
# 'duration': duration,
# 'max_kp': max_kp,
# 'time_steps': time_steps
event_id = "G5H3-202405101500"
event_start = datetime.datetime(2024, 5, 10, 15, 0).isoformat()
event_end = datetime.datetime(2024, 5, 10, 18, 0).isoformat()
duration = datetime.datetime.fromisoformat(event_end) - datetime.datetime.fromisoformat(event_start)
max_kp = 9.0
time_steps = int(duration.total_seconds() / (15*60))
catalog = {}
catalog[event_id] = {
  'date_start': event_start,
  'date_end': event_end,
  'duration': duration,
  'max_kp': max_kp,
  'time_steps': time_steps
}

event_catalog = EventCatalog(catalog=catalog)
valid_event_id = ["G5H3-202405101500"]

date_exclusions = []
datasets_omniweb_valid = []
datasets_jpld_valid = []
datasets_sunmoon_valid = []

# Process validation events
for event_id in valid_event_id:
    print('Excluding event ID: {}'.format(event_id))

    if event_id not in event_catalog:
        raise ValueError('Event ID {} not found in EventCatalog'.format(event_id))

    event = event_catalog[event_id]
    exclusion_start = datetime.datetime.fromisoformat(event['date_start']) - datetime.timedelta(minutes=context_window * delta_minutes)
    exclusion_end = datetime.datetime.fromisoformat(event['date_end'])
    date_exclusions.append((exclusion_start, exclusion_end))
    print('Exclusion start: {}, end: {}'.format(exclusion_start, exclusion_end))

    datasets_omniweb_valid.append(OMNIWeb(dataset_omniweb_dir, date_start=exclusion_start, date_end=exclusion_end, return_as_image_size=image_size))
    datasets_jpld_valid.append(JPLD(dataset_jpld_dir, date_start=exclusion_start, date_end=exclusion_end))
    datasets_sunmoon_valid.append(SunMoonGeometry(date_start=exclusion_start, date_end=exclusion_end))

# Create validation dataset
dataset_jpld_valid = Union(datasets=datasets_jpld_valid)
dataset_omniweb_valid = Union(datasets=datasets_omniweb_valid)
dataset_sunmoon_valid = Union(datasets=datasets_sunmoon_valid)

# Create training dataset excluding validation events
dataset_jpld_train = JPLD(dataset_jpld_dir, date_start=date_start, date_end=date_end, date_exclusions=date_exclusions)
dataset_omniweb_train = OMNIWeb(dataset_omniweb_dir, date_start=date_start, date_end=date_end, date_exclusions=date_exclusions, return_as_image_size=image_size)
dataset_sunmoon_train = SunMoonGeometry(date_start=date_start, date_end=date_end, date_exclusions=date_exclusions)

# Create Sequence datasets
dataset_valid = Sequences(datasets=[dataset_jpld_valid, dataset_omniweb_valid, dataset_sunmoon_valid], sequence_length=training_sequence_length, dilation=date_dilation, delta_minutes=delta_minutes)
dataset_train = Sequences(datasets=[dataset_jpld_train, dataset_omniweb_train, dataset_sunmoon_train], sequence_length=training_sequence_length, dilation=date_dilation, delta_minutes=delta_minutes)


# Q: What is one question that you have answered using these data? 

This dataset addresses the temporal relationship between ionospheric drivers (solar wind, interplanetary magnetic field strength, Kp, etc.) and Global Navigation Satellite Systems  derived global ionospheric maps (GIMs). It does this by integrating heterogeneous, multi-source observations and harmonizing temporal and spatial scales. This dataset provides the foundation upon which advanced machine learning architectures can be developed, tested, and benchmarked for global-scale ionospheric nowcasting and forecasting.

# Q: What is one unanswered question that you think could be answered using these data? 

An unanswered question that remains from this dataset is the physical relationships between the various ionospheric drivers and GIMs. This dataset was used to train a machine learning model to forecast GIMs, but not to analyze the relationships between drivers and the resulting GIM. It would be interesting to in future address more than just the temporal relationships between these data.